# 3 · Variance analysis — how the repertoires spreadBeyond *separability* (notebook 02), a descriptive question: how is the **variance** of the repertoiresdistributed? Two complementary views:- **A — within-group dispersion:** how spread out are the repertoires of each group around their  centroid? (is one group more heterogeneous between patients?)- **C — PCA / explained variance:** where does the variation come from, and how do the groups sit?Labels: cell type (CD4/CD8) and cohort (HD/MS/T1D).

## SetupOne mean+cov descriptor per repertoire, plus cell-type and cohort labels.

In [ ]:
import sys, re, osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltREPERTOIRE_DIR = '../scripts/repertoire'CLOUDS_DIR     = '../data/clouds_embedded_TRB'sys.path.insert(0, REPERTOIRE_DIR)import rep_data, rep_descriptors as rdEMB = rep_data.EMB_COLSdef parse(stem):    pid = re.match(r'(MS\d+|T1D\d+|HD_[A-Za-z]+)', stem).group(1)    cell = 'CD4' if 'CD4' in stem else 'CD8'    cohort = 'MS' if stem.startswith('MS') else ('T1D' if stem.startswith('T1D') else 'HD')    return pid, cell, cohortfiles = rep_data.cloud_files(CLOUDS_DIR)V, cell_lab, coh_lab = [], [], []for stem, path in files.items():    Z, w, _ = rep_data.load_cloud(path, cols=('w_log',))    Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)    V.append(rd.mean_cov_weighted_np(Zn, w / w.sum()))    pid, cell, cohort = parse(stem)    cell_lab.append(cell); coh_lab.append(cohort)V = np.vstack(V); cell_lab = np.array(cell_lab); coh_lab = np.array(coh_lab)group_lab = np.array([f'{c}-{t}' for c, t in zip(coh_lab, cell_lab)])print('clouds:', V.shape[0])print('by cell:', pd.Series(cell_lab).value_counts().to_dict())print('by cohort:', pd.Series(coh_lab).value_counts().to_dict())

## A · Within-group dispersionFor each group, measure how far each repertoire sits from its group centroid (cosine distance). Highermedian = more heterogeneous group. Boxplots show the distribution; the box is the middle 50% ofrepertoires, the orange line the median, and points beyond the whiskers are outliers (1.5×IQR rule).**Insight:** CD8 repertoires are more dispersed than CD4 (median 0.037 vs 0.027) — patients' CD8 cloudsresemble each other less. The pattern holds across all three cohorts, consistent with CD8 being moreclonal and idiosyncratic per individual. T1D-CD4 is the most dispersed CD4 group (0.034 vs 0.023 in HD),a possible autoimmunity signal.

In [ ]:
def within_dispersion(V, labels):    Vn = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-8)    out = {}    for g in np.unique(labels):        X = Vn[labels==g]        c = X.mean(0); c /= (np.linalg.norm(c)+1e-8)        out[g] = 1 - X @ c   # cosine distance of each member to its group centroid    return outfig, axes = plt.subplots(1, 2, figsize=(15, 5.5))disp_cell = within_dispersion(V, cell_lab)axes[0].boxplot([disp_cell[g] for g in ['CD4','CD8']], labels=['CD4','CD8'])axes[0].set_ylabel('Cosine distance to group centroid', fontsize=12, fontweight='bold')axes[0].set_title('Within-group dispersion by cell type', fontsize=12); axes[0].grid(True, alpha=0.25)order = ['HD-CD4','HD-CD8','MS-CD4','MS-CD8','T1D-CD4','T1D-CD8']disp_g = within_dispersion(V, group_lab); order = [g for g in order if g in disp_g]axes[1].boxplot([disp_g[g] for g in order], labels=order)axes[1].set_ylabel('Cosine distance to group centroid', fontsize=12, fontweight='bold')axes[1].set_title('Within-group dispersion by cohort + cell', fontsize=12)axes[1].tick_params(axis='x', rotation=45); axes[1].grid(True, alpha=0.25)plt.suptitle('Within-group dispersion — how heterogeneous is each group? (higher = more spread)', fontsize=13, fontweight='bold')plt.tight_layout(); plt.show()print('median dispersion:')for g in ['CD4','CD8']: print(f'   {g}: {np.median(disp_cell[g]):.4f}')for g in order: print(f'   {g}: {np.median(disp_g[g]):.4f}')

## C · PCA / explained varianceProject the 8384-d descriptors to 2D via PCA (colour = cell type, marker = cohort), and show how muchvariance each principal component captures.**Insight:** PC1 alone captures ~44% of all variation, and it separates CD4 from CD8 cleanly — thedominant source of variance *is* the cell lineage. Cohorts (HD/MS/T1D) mix within each colour → nocohort structure. CD8 points are more spread than CD4 (echoing view A).

In [ ]:
from sklearn.decomposition import PCAVn = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-8)pca = PCA(n_components=10).fit(Vn)P = pca.transform(Vn)fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))colors = {'CD4':'#2C7FB8', 'CD8':'#C0392B'}; markers = {'HD':'o', 'MS':'s', 'T1D':'^'}for cell in ['CD4','CD8']:    for coh in ['HD','MS','T1D']:        m = (cell_lab==cell) & (coh_lab==coh)        if m.sum()==0: continue        axes[0].scatter(P[m,0], P[m,1], c=colors[cell], marker=markers[coh], s=50, alpha=0.7,                        edgecolor='white', label=f'{coh}-{cell}')axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=12, fontweight='bold')axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=12, fontweight='bold')axes[0].set_title('PCA of descriptors (colour=cell, marker=cohort)', fontsize=12)axes[0].legend(fontsize=8, loc='best'); axes[0].grid(True, alpha=0.25)cum = np.cumsum(pca.explained_variance_ratio_)*100axes[1].bar(range(1,11), pca.explained_variance_ratio_*100, alpha=0.7, color='#065A82', label='per component')axes[1].plot(range(1,11), cum, marker='o', color='#C0392B', label='cumulative')axes[1].set_xlabel('Principal component', fontsize=12, fontweight='bold')axes[1].set_ylabel('Variance explained (%)', fontsize=12, fontweight='bold')axes[1].set_title('Explained variance', fontsize=12)axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.25)plt.suptitle('PCA and explained variance', fontsize=13, fontweight='bold')plt.tight_layout(); plt.show()print('variance explained by PC1-3:', (pca.explained_variance_ratio_[:3]*100).round(1))

## Summary- **Within-group dispersion (A):** CD8 > CD4 — CD8 repertoires are more heterogeneous between patients,  consistently across cohorts.- **PCA (C):** PC1 (~44% of variance) is the CD4/CD8 axis; cohorts do not form separate clusters.Both views agree: the dominant, structured source of variance among repertoires is the **CD4/CD8lineage**, and CD8 is the more variable class. Disease cohort is not a major axis of repertoirevariance at this (global, whole-cloud) level.